# Calculate effective coverage by quintile and scenario

Also by age, sex, and pregnancy status (though coverage will not vary by pregnancy status due to a lack of data).

This is similar to what the pregnancy simulation does at the individual level, but using groups instead.
It can be shared between multiplication models that do not incorporate individual heterogeneity.

In [1]:
import pandas as pd

In [2]:
location = "nigeria"
fortificant = "iron"

In [3]:
# Parameters
location = "india"
fortificant = "folate"


In [4]:
results_dir = f'../results/{fortificant}'

In [5]:
full_coverage_probability = pd.read_csv(f'{results_dir}/baseline_fortification/full_coverage/{location}.csv')
full_coverage_probability = full_coverage_probability.set_index([c for c in full_coverage_probability.columns if c != 'value']).value
full_coverage_probability

sex     age_start  age_end  wealth_quintile  vehicle_name
female  0.0        5.0      lowest           rice            0.156303
                            second           rice            0.173319
                            middle           rice            0.172177
                            fourth           rice            0.123566
                            highest          rice            0.062838
        5.0        15.0     lowest           rice            0.177188
                            second           rice            0.198127
                            middle           rice            0.182862
                            fourth           rice            0.146883
                            highest          rice            0.077928
        15.0       30.0     lowest           rice            0.174926
                            second           rice            0.205154
                            middle           rice            0.183202
                            four

In [6]:
any_coverage_probability = pd.read_csv(f'{results_dir}/baseline_fortification/any_coverage/{location}.csv')
any_coverage_probability = any_coverage_probability.set_index([c for c in any_coverage_probability.columns if c != 'value']).value
any_coverage_probability

sex     age_start  age_end  wealth_quintile  vehicle_name
female  0.0        5.0      lowest           rice            0.560845
                            second           rice            0.535335
                            middle           rice            0.492510
                            fourth           rice            0.454610
                            highest          rice            0.293520
        5.0        15.0     lowest           rice            0.713577
                            second           rice            0.671205
                            middle           rice            0.624088
                            fourth           rice            0.568540
                            highest          rice            0.360006
        15.0       30.0     lowest           rice            0.642256
                            second           rice            0.612181
                            middle           rice            0.545258
                            four

In [7]:
partial_coverage_mean = pd.read_csv(f'{results_dir}/baseline_fortification/partial_coverage_amount/mean/{location}.csv')
partial_coverage_mean = partial_coverage_mean.set_index([c for c in partial_coverage_mean.columns if c != 'value']).value
partial_coverage_mean

sex     age_start  age_end  wealth_quintile  vehicle_name
female  0.0        5.0      lowest           rice            0.524382
                            second           rice            0.552721
                            middle           rice            0.551125
                            fourth           rice            0.530210
                            highest          rice            0.471885
        5.0        15.0     lowest           rice            0.566689
                            second           rice            0.565275
                            middle           rice            0.560457
                            fourth           rice            0.535734
                            highest          rice            0.477592
        15.0       30.0     lowest           rice            0.535427
                            second           rice            0.563819
                            middle           rice            0.558023
                            four

In [8]:
current_coverage = full_coverage_probability + (any_coverage_probability - full_coverage_probability) * partial_coverage_mean
current_coverage

sex     age_start  age_end  wealth_quintile  vehicle_name
female  0.0        5.0      lowest           rice            0.368437
                            second           rice            0.373413
                            middle           rice            0.348721
                            fourth           rice            0.299089
                            highest          rice            0.171693
        5.0        15.0     lowest           rice            0.481154
                            second           rice            0.465546
                            middle           rice            0.430151
                            fourth           rice            0.372779
                            highest          rice            0.212646
        15.0       30.0     lowest           rice            0.425147
                            second           rice            0.434643
                            middle           rice            0.385238
                            four

In [9]:
scenarios = {
    "india": ["intervention"],
    "nigeria": ["intervention"],
    "ethiopia": ["intervention_25_nrv", "intervention_100_nrv"],
}[location]

In [10]:
fortifiability = pd.read_csv(f'{results_dir}/../vehicle_consumption/fortifiability/{location}.csv')
fortifiability = fortifiability.set_index([c for c in fortifiability.columns if c != 'value']).value
fortifiability

sex     age_start  age_end  wealth_quintile  vehicle_name
female  0.0        5.0      lowest           rice            0.750635
                            second           rice            0.765040
                            middle           rice            0.758213
                            fourth           rice            0.726724
                            highest          rice            0.648106
        5.0        15.0     lowest           rice            0.809506
                            second           rice            0.806553
                            middle           rice            0.787083
                            fourth           rice            0.751067
                            highest          rice            0.659888
        15.0       30.0     lowest           rice            0.765627
                            second           rice            0.783189
                            middle           rice            0.761376
                            four

In [11]:
import pathlib

for scenario in scenarios:
    intervention_coverage = pd.read_csv(f'{results_dir}/{scenario}/intervention_fortification/any_coverage/{location}.csv')
    intervention_coverage = intervention_coverage.set_index([c for c in intervention_coverage.columns if c != 'value']).value
    target_coverage = intervention_coverage * fortifiability
    display(target_coverage)
    assert (target_coverage > current_coverage.reindex_like(target_coverage)).all()
    # Not all coverage is effective -- this is as a proportion of coverage!
    effective_coverage = pd.read_csv(f'{results_dir}/{scenario}/intervention_fortification/effective_coverage/{location}.csv')
    effective_coverage = effective_coverage.set_index([c for c in effective_coverage.columns if c != 'value']).value
    effective_intervention_coverage = target_coverage * effective_coverage
    path = f'{results_dir}/{scenario}/effective_intervention_coverage/{location}.csv'
    pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
    effective_intervention_coverage.reset_index().to_csv(path, index=False)


sex     age_start  age_end  wealth_quintile  vehicle_name
female  0.0        5.0      lowest           rice            0.600508
                            second           rice            0.612032
                            middle           rice            0.606570
                            fourth           rice            0.581379
                            highest          rice            0.518485
        5.0        15.0     lowest           rice            0.647605
                            second           rice            0.645243
                            middle           rice            0.629666
                            fourth           rice            0.600854
                            highest          rice            0.527910
        15.0       30.0     lowest           rice            0.612502
                            second           rice            0.626551
                            middle           rice            0.609101
                            four

In [12]:
effective_baseline_coverage = current_coverage * effective_coverage
effective_baseline_coverage

sex     age_start  age_end  wealth_quintile  vehicle_name
female  0.0        5.0      lowest           rice            0.294750
                            second           rice            0.298731
                            middle           rice            0.278976
                            fourth           rice            0.239271
                            highest          rice            0.137355
        5.0        15.0     lowest           rice            0.384923
                            second           rice            0.372437
                            middle           rice            0.344120
                            fourth           rice            0.298223
                            highest          rice            0.170117
        15.0       30.0     lowest           rice            0.340118
                            second           rice            0.347715
                            middle           rice            0.308190
                            four

In [13]:
path = f'{results_dir}/effective_baseline_coverage/{location}.csv'
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
effective_baseline_coverage.reset_index().to_csv(path, index=False)